# NanoQuant · Qwen3.8-27B · 0.55 bpw · Google Colab

Этот ноутбук клонирует [твой форк NanoQuant](https://github.com/rkuznetsovcg/NanoQuant), ставит нужные зависимости, берёт подготовленный калибровочный набор, запускает профиль `qwen38-0.55-balanced` с равномерными рангами и сохраняет итоговый `.pt` в Google Drive.

**Как запустить:** в Colab выбери `Среда выполнения → Сменить тип среды выполнения → GPU` и доступный GPU с памятью около 80 ГБ (H100 предпочтителен). Затем выбери `Среда выполнения → Выполнить все` и подтверди доступ к Google Drive. Ячейки проверят, хватает ли памяти и диска, прежде чем скачивать веса модели.

После квантования отдельная ячейка последовательно оценивает исходную BF16-модель и NanoQuant-чекпойнт с одинаковыми токенизатором, ревизией модели, длиной контекста, корпусами, задачами и настройками. Она отдельно показывает PPL, KL-дивергенцию и точность и сохраняет полный парный отчёт; повторной квантизации не запускает.

Возобновляющие снимки блоков по умолчанию пишутся на локальный диск Colab ради скорости: Drive медленный для этих многогигабайтных временных файлов. Если сама VM Colab будет уничтожена, локальные снимки пропадут. Для сохранения снимков между VM включи `PERSIST_RESUME_ON_DRIVE` в следующей ячейке; это может заметно увеличить время записи.


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/rkuznetsovcg/NanoQuant.git"
REPO_BRANCH = "main"
MODEL_ID = "Qwen/Qwen3.8-27B"
PROFILE_RELATIVE_PATH = "configs/qwen38-0.55-balanced.json"
RUN_NAME = "qwen38-27b-0.55-balanced-uniform-bounded"

# False ускоряет запись промежуточных чекпойнтов. Поставь True до первого запуска,
# если нужно переживать полное уничтожение Colab VM; Drive будет получать крупные снимки блоков.
PERSIST_RESUME_ON_DRIVE = False

# Защита от старта на слишком маленьком runtime. Не снижай пороги, если не уверен в памяти.
MIN_GPU_VRAM_GIB = 70.0
MIN_HOST_RAM_TOTAL_GIB = 70.0
MIN_HOST_RAM_AVAILABLE_GIB = 65.0
MIN_FREE_DISK_GIB = 120.0

ENABLE_FLASHQLA_WHEN_COMPATIBLE = True
DRIVE_MOUNT = Path("/content/drive")
DRIVE_RUN_DIR = DRIVE_MOUNT / "MyDrive" / "NanoQuant" / "runs" / RUN_NAME
REPO_DIR = Path("/content/NanoQuant")
LOCAL_RUN_DIR = Path("/content/nanoquant") / RUN_NAME
LOCAL_OUTPUT = LOCAL_RUN_DIR / "Qwen3.8-27B-NQ-0.55-balanced-uniform-bounded.pt"


In [ ]:
import os
import re
import shutil
from pathlib import Path

import torch


def gib(value):
    return value / (1024 ** 3)


def read_meminfo_gib(label):
    for line in Path("/proc/meminfo").read_text().splitlines():
        if line.startswith(label + ":"):
            return int(line.split()[1]) * 1024 / (1024 ** 3)
    raise RuntimeError(f"Не удалось прочитать {label} из /proc/meminfo")


if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU не найден. В Colab выбери Runtime → Change runtime type → GPU и запусти всё снова.")

props = torch.cuda.get_device_properties(0)
gpu_vram_gib = gib(props.total_memory)
host_total_gib = read_meminfo_gib("MemTotal")
host_available_gib = read_meminfo_gib("MemAvailable")
disk_free_gib = gib(shutil.disk_usage("/content").free)
capability = torch.cuda.get_device_capability(0)
cuda_version = torch.version.cuda or "нет CUDA runtime"

def version_tuple(value):
    return tuple(int(part) for part in re.findall(r"\d+", value.split("+", 1)[0])[:3])

flashqla_compatible = (
    ENABLE_FLASHQLA_WHEN_COMPATIBLE
    and capability >= (9, 0)
    and version_tuple(torch.__version__) >= (2, 8)
    and version_tuple(cuda_version) >= (12, 8)
)

print(f"GPU: {props.name} | {gpu_vram_gib:.1f} GiB | compute {capability[0]}.{capability[1]}")
print(f"PyTorch {torch.__version__} | CUDA runtime {cuda_version}")
print(f"RAM: {host_available_gib:.1f} GiB available / {host_total_gib:.1f} GiB total")
print(f"Free /content disk: {disk_free_gib:.1f} GiB")
print(f"FlashQLA install profile: {'eligible' if flashqla_compatible else 'standard Qwen fast kernels'}")

problems = []
if gpu_vram_gib < MIN_GPU_VRAM_GIB:
    problems.append(f"нужно не менее {MIN_GPU_VRAM_GIB:.0f} GiB GPU VRAM")
if host_total_gib < MIN_HOST_RAM_TOTAL_GIB:
    problems.append(f"нужно не менее {MIN_HOST_RAM_TOTAL_GIB:.0f} GiB общей RAM: BF16-веса сначала загружаются в RAM")
if host_available_gib < MIN_HOST_RAM_AVAILABLE_GIB:
    problems.append(f"нужно не менее {MIN_HOST_RAM_AVAILABLE_GIB:.0f} GiB свободной RAM")
if disk_free_gib < MIN_FREE_DISK_GIB:
    problems.append(f"нужно не менее {MIN_FREE_DISK_GIB:.0f} GiB свободного диска под веса и промежуточные блоки")
if problems:
    raise RuntimeError("Этот runtime мал для выбранного запуска: " + "; ".join(problems))


## Подключение Google Drive

Drive нужен для итогового чекпойнта и файла с настройками запуска. Саму модель и временную работу NanoQuant хранит в диске Colab, чтобы не читать и писать веса через сетевую файловую систему Drive.


In [ ]:
from google.colab import drive

DRIVE_MOUNT.mkdir(parents=True, exist_ok=True)
drive.mount(str(DRIVE_MOUNT))
DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"Итоговые файлы будут сохранены в: {DRIVE_RUN_DIR}")


## Получение форка и калибровочного набора

Набор уже лежит в GitHub-репозитории, поэтому здесь не нужно отдельно скачивать датасеты или собирать их заново.


In [ ]:
import os
import subprocess
import sys


def run_command(command, cwd=None, env=None):
    print("$", " ".join(map(str, command)))
    command_env = os.environ.copy() if env is None else env.copy()
    source_dir = REPO_DIR / "src"
    if source_dir.is_dir():
        old_pythonpath = command_env.get("PYTHONPATH", "")
        command_env["PYTHONPATH"] = str(source_dir) + (os.pathsep + old_pythonpath if old_pythonpath else "")
    subprocess.run(command, cwd=cwd, env=command_env, check=True)


if not (REPO_DIR / ".git").exists():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    run_command(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
else:
    run_command(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL])
    run_command(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH])
    run_command(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH])
    run_command(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH])

PROFILE_PATH = REPO_DIR / PROFILE_RELATIVE_PATH
CALIBRATION_PATH = REPO_DIR / "data/calibration/generated/qwen3.8-27b-multidomain-128x2048/dataset"
if not PROFILE_PATH.is_file():
    raise FileNotFoundError(f"В форке нет профиля: {PROFILE_PATH}")
if not (CALIBRATION_PATH / "dataset_info.json").is_file():
    raise FileNotFoundError(f"В форке нет калибровочного набора: {CALIBRATION_PATH}")

# Keep source imports available even if Colab's editable-install path is not refreshed.
source_dir = str(REPO_DIR / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)

GIT_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
print(f"NanoQuant commit: {GIT_COMMIT}")
print(f"Профиль: {PROFILE_PATH}")
print(f"Калибровка: {CALIBRATION_PATH}")


## Установка CUDA-зависимостей

Для H100 ноутбук сначала пробует совместимый профиль с FlashQLA для Gated DeltaNet; если установка не проходит, он автоматически ставит стандартные быстрые Qwen-ядра и оставляет линейное внимание на FLA. FlashQLA использует тот же BF16-путь модели и включается только на совместимой связке GPU, CUDA и PyTorch.


In [ ]:
import os
import sys


def install_extra(extra):
    command = [sys.executable, "-m", "pip", "install", "--no-build-isolation", "--no-cache-dir", "-e", f".[{extra}]"]
    print("$", " ".join(command))
    return subprocess.run(command, cwd=REPO_DIR, env=os.environ.copy()).returncode


if flashqla_compatible:
    status = install_extra("qwen-flashqla")
    if status != 0:
        print("FlashQLA не собрался в этом runtime; включаю резервный путь FLA Triton + kernels.")
        os.environ["FLA_FLASH_QLA"] = "0"
        status = install_extra("qwen-fast,kernels")
else:
    status = install_extra("qwen-fast,kernels")

if status != 0:
    raise RuntimeError("Не удалось установить зависимости NanoQuant. См. вывод pip выше.")
print("Зависимости установлены.")


## Проверка модели, зависимостей и калибровки

Эта ячейка загружает только конфигурацию модели и локальный небольшой калибровочный набор. Полные 27B BF16-веса начнут скачиваться только в следующей ячейке.


In [ ]:
import json
import sys
from datasets import load_from_disk
from transformers import AutoConfig

# The project uses a src/ layout; add it explicitly for this notebook kernel.
source_dir = str(REPO_DIR / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)

from nanoquant.utils.attention_utils import resolve_attention_backend
from nanoquant.utils.load_utils import _check_fast_linear_attention

profile = json.loads(PROFILE_PATH.read_text())
if profile["model_id"] != MODEL_ID:
    raise ValueError(f"Профиль указывает неожиданную модель: {profile['model_id']}")
if profile["bits"] != 0.55 or profile["num_calib_samples"] != 128 or profile["seqlen"] != 2048:
    raise ValueError("Профиль больше не совпадает с запланированным запуском 0.55 bpw / 128 × 2048")

import re
from huggingface_hub import HfApi, try_to_load_from_cache


def resolve_hub_revision(model_id, requested_revision="main"):
    """Resolve an immutable Hub commit SHA, including from an existing local snapshot."""
    hub_error = None
    try:
        commit_sha = HfApi().model_info(model_id, revision=requested_revision).sha
        if commit_sha:
            return commit_sha, "Hugging Face Hub model_info"
        hub_error = RuntimeError("Hugging Face Hub returned no commit SHA")
    except Exception as error:
        hub_error = error

    cached_config = try_to_load_from_cache(model_id, "config.json", revision=requested_revision)
    if isinstance(cached_config, str):
        cached_sha = Path(cached_config).resolve().parent.name
        if re.fullmatch(r"[0-9a-f]{40,64}", cached_sha):
            return cached_sha, "local Hugging Face snapshot cache"

    raise RuntimeError(
        f"Не удалось разрешить commit SHA для {model_id}@{requested_revision}; "
        f"Hub: {type(hub_error).__name__}: {hub_error}; локальный snapshot не найден"
    ) from hub_error


MODEL_REVISION, MODEL_REVISION_SOURCE = resolve_hub_revision(MODEL_ID)
model_config = AutoConfig.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, trust_remote_code=True,
)
config_revision = getattr(model_config, "_commit_hash", None)
if config_revision and config_revision != MODEL_REVISION:
    raise RuntimeError(
        f"Конфиг загружен с неожиданной ревизии: {config_revision} != {MODEL_REVISION}"
    )
if model_config.model_type not in {"qwen3_5", "qwen3_5_text"}:
    raise RuntimeError(f"Неожиданный тип модели: {model_config.model_type}")
_check_fast_linear_attention(model_config, required=True)
attention_backend = resolve_attention_backend(model_config, profile.get("attn_implementation", "auto"))

calibration = load_from_disk(str(CALIBRATION_PATH))
if len(calibration) < profile["num_calib_samples"]:
    raise ValueError(f"В датасете только {len(calibration)} строк, нужно {profile['num_calib_samples']}")
first_window = calibration[0]["input_ids"]
if len(first_window) != profile["seqlen"]:
    raise ValueError(f"Длина окна {len(first_window)}, ожидалось {profile['seqlen']}")

LOCAL_RUN_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
resume_path = (DRIVE_RUN_DIR / "resume") if PERSIST_RESUME_ON_DRIVE else (LOCAL_RUN_DIR / "resume")
resume_path.mkdir(parents=True, exist_ok=True)

profile["model_id"] = MODEL_ID
profile["model_revision"] = MODEL_REVISION
profile["calib_dataset"] = str(CALIBRATION_PATH)
profile["qmodel_path"] = str(LOCAL_OUTPUT)
profile["resume_dir"] = str(resume_path)
profile["attn_implementation"] = "auto"
CONFIG_PATH = LOCAL_RUN_DIR / "qwen38-0.55-balanced-uniform-bounded-colab.json"
CONFIG_PATH.write_text(json.dumps(profile, indent=2) + "\n")

print(f"Ревизия исходной модели: {MODEL_REVISION} ({MODEL_REVISION_SOURCE})")
print(f"Модель: {model_config.model_type}; параметры: {model_config.text_config.num_hidden_layers if hasattr(model_config, 'text_config') else model_config.num_hidden_layers} слоёв")
print(f"Полное внимание: {attention_backend}")
print(f"Калибровка: {len(calibration)} окон; по {len(first_window)} токенов")
print(f"Resume: {resume_path}")
print(f"Промежуточная конфигурация: {CONFIG_PATH}")


## Запуск квантования и сохранение результата

Запуск использует тот же профиль из репозитория. Если процесс прервётся, но Colab VM останется живой, снова выполни последнюю ячейку: NanoQuant продолжит с сохранённого блока. Под ячейкой будет обновляться панель текущего этапа, прогресс-блоков, занятость GPU и последние сообщения. Полный лог сохранится в Drive; при сбое ячейка покажет настоящую причину из процесса. После успешного завершения ноутбук скопирует чекпойнт и метаданные в Drive.

Если появится ошибка о нехватке системной RAM, нужен runtime с большим объёмом **обычной RAM**, даже если у GPU 80 ГБ VRAM. Веса этой модели в BF16 сначала загружаются в системную память.


In [ ]:
import codecs
import datetime
import errno
import fcntl
import html
import json
import os
import pty
import re
import selectors
import shutil
import struct
import subprocess
import sys
import termios
import time
from collections import deque
from IPython.display import FileLink, HTML, display

DRIVE_OUTPUT = DRIVE_RUN_DIR / LOCAL_OUTPUT.name
DRIVE_COMPLETE = DRIVE_RUN_DIR / "run_complete.json"
DRIVE_TEMP_OUTPUT = DRIVE_RUN_DIR / (LOCAL_OUTPUT.name + ".uploading")
LOCAL_LOG = LOCAL_RUN_DIR / "nanoquant.log"
DRIVE_LOG = DRIVE_RUN_DIR / "nanoquant.log"
ANSI_RE = re.compile(r"\x1b(?:\[[0-?]*[ -/]*[@-~]|\][^\x07]*(?:\x07|\x1b\\))")


def run_nanoquant_live(command, cwd, log_path):
    state = {"stage": "Запуск NanoQuant", "progress": None, "line": "Подключаю вывод процесса…"}
    recent = deque(maxlen=8)
    started = time.monotonic()

    def consume_line(raw_line):
        line = ANSI_RE.sub("", raw_line).strip()
        if not line:
            return
        state["line"] = line
        recent.append(line)
        lower = line.lower()
        if "loading model" in lower or "downloading" in lower:
            state["stage"] = "Загрузка весов модели"
            state["progress"] = None
        elif "calibration strategy" in lower or "single pass:" in lower:
            state["stage"] = "Сбор статистики калибровки"
            state["progress"] = None
        elif "caching initial inputs" in lower:
            state["stage"] = "Кеширование входов блоков"
            state["progress"] = None
        elif "compressing layers" in lower:
            state["stage"] = "Квантование и реконструкция блоков"
        elif "tuning non-factorized" in lower:
            state["stage"] = "Настройка нефакторизованных весов"
        elif "initialization via admm" in lower:
            state["stage"] = "Факторизация весов (ADMM)"
        elif "tuning factorized" in lower:
            state["stage"] = "Настройка факторизованных весов"
        elif "caching teacher hidden states" in lower or "model-level kd" in lower:
            state["stage"] = "Финальная дистилляция модели"
        elif "saved quantized model" in lower:
            state["stage"] = "Сохранение чекпойнта"

        match = re.search(r"(\d{1,3})%\|.*?\|\s*(\d+)\s*/\s*(\d+)", line)
        if match:
            state["progress"] = (int(match.group(2)), int(match.group(3)))
        elif state["stage"] == "Сбор статистики калибровки":
            match = re.search(r"\|\s*(\d+)\s*/\s*(\d+)", line)
            if match:
                state["progress"] = (int(match.group(1)), int(match.group(2)))

    def render_panel(status="Работает"):
        elapsed = int(time.monotonic() - started)
        hours, remainder = divmod(elapsed, 3600)
        minutes, seconds = divmod(remainder, 60)
        progress_html = ""
        progress = state["progress"]
        if progress and progress[1] > 0:
            current, total = progress
            percent = 100 * current / total
            progress_html = (
                f'<progress value="{current}" max="{total}" '
                'style="width:100%;height:18px"></progress>'
                f'<div>{current} / {total} · {percent:.1f}%</div>'
            )
        try:
            free_vram, total_vram = torch.cuda.mem_get_info(0)
            gpu_text = f"{(total_vram-free_vram)/(1024**3):.1f} / {total_vram/(1024**3):.1f} GiB"
        except Exception:
            gpu_text = "считывание недоступно"
        try:
            free_disk = shutil.disk_usage("/content").free / (1024**3)
            disk_text = f"свободно {free_disk:.1f} GiB"
        except Exception:
            disk_text = "считывание недоступно"
        tail = html.escape("\n".join(recent)[-5000:])
        current_line = html.escape(state["line"][-500:])
        return HTML(
            '<div style="border:1px solid #9aa4b2;border-radius:10px;padding:14px;'
            'font:14px system-ui;max-width:1000px">'
            f'<div style="font-size:18px;font-weight:700">NanoQuant · {html.escape(status)}</div>'
            f'<div style="margin:8px 0"><b>Этап:</b> {html.escape(state["stage"])}</div>'
            f'{progress_html}'
            f'<div style="margin:8px 0;color:#52606d">Время: {hours:02d}:{minutes:02d}:{seconds:02d} '
            f'· GPU: {gpu_text} · Диск: {disk_text}</div>'
            f'<div style="margin:8px 0;overflow-wrap:anywhere">{current_line}</div>'
            f'<details><summary>Последние сообщения процесса</summary>'
            f'<pre style="white-space:pre-wrap;max-height:260px;overflow:auto">{tail}</pre></details>'
            '</div>'
        )

    env = os.environ.copy()
    source_dir = str(Path(cwd) / "src")
    previous_pythonpath = env.get("PYTHONPATH", "")
    env["PYTHONPATH"] = source_dir + (os.pathsep + previous_pythonpath if previous_pythonpath else "")
    env["PYTHONUNBUFFERED"] = "1"
    env["TQDM_MININTERVAL"] = "1"
    env["TQDM_MINITERS"] = "1"
    env.setdefault("TERM", "xterm-256color")

    master_fd, slave_fd = pty.openpty()
    fcntl.ioctl(slave_fd, termios.TIOCSWINSZ, struct.pack("HHHH", 32, 140, 0, 0))
    process = subprocess.Popen(
        command, cwd=cwd, env=env, stdin=subprocess.DEVNULL,
        stdout=slave_fd, stderr=subprocess.STDOUT, close_fds=True,
    )
    os.close(slave_fd)
    os.set_blocking(master_fd, False)
    selector = selectors.DefaultSelector()
    selector.register(master_fd, selectors.EVENT_READ)
    decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
    pending = ""
    last_update = 0.0
    display_handle = display(render_panel(), display_id=True)

    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("wb") as log_file:
        while process.poll() is None or selector.get_map():
            events = selector.select(timeout=0.5) if selector.get_map() else []
            for key, _ in events:
                try:
                    chunk = os.read(key.fd, 65536)
                except BlockingIOError:
                    continue
                except OSError as error:
                    if error.errno in (errno.EIO, errno.EBADF):
                        selector.unregister(key.fd)
                        continue
                    raise
                if not chunk:
                    selector.unregister(key.fd)
                    continue
                log_file.write(chunk)
                log_file.flush()
                pending += decoder.decode(chunk)
                while True:
                    separators = [position for position in (pending.find("\r"), pending.find("\n")) if position >= 0]
                    if not separators:
                        break
                    position = min(separators)
                    consume_line(pending[:position])
                    pending = pending[position + 1:]
            now = time.monotonic()
            if not selector.get_map() and process.poll() is None:
                time.sleep(0.2)
            if now - last_update >= 0.8:
                display_handle.update(render_panel())
                last_update = now

        pending += decoder.decode(b"", final=True)
        if pending.strip():
            consume_line(pending)

    selector.close()
    os.close(master_fd)
    return_code = process.wait()
    display_handle.update(render_panel("Завершено" if return_code == 0 else f"Ошибка · код {return_code}"))
    return return_code


if DRIVE_COMPLETE.is_file() and DRIVE_OUTPUT.is_file() and DRIVE_OUTPUT.stat().st_size > 0:
    print(f"Готовый запуск уже сохранён: {DRIVE_OUTPUT}")
    print(f"Метка завершения: {DRIVE_COMPLETE}")
else:
    if LOCAL_OUTPUT.exists():
        LOCAL_OUTPUT.unlink()

    command = [sys.executable, "-m", "nanoquant.main", str(CONFIG_PATH)]
    exit_code = run_nanoquant_live(command, REPO_DIR, LOCAL_LOG)
    try:
        shutil.copy2(LOCAL_LOG, DRIVE_LOG)
    except Exception as copy_error:
        print(f"Не удалось скопировать лог в Drive: {copy_error}")

    if exit_code != 0:
        raw_tail = LOCAL_LOG.read_bytes()[-24000:].decode("utf-8", errors="replace")
        raw_tail = ANSI_RE.sub("", raw_tail).replace("\r", "\n")
        tail_lines = [line for line in raw_tail.splitlines() if line.strip()][-35:]
        display(FileLink(str(LOCAL_LOG), result_html_prefix="Полный лог: "))
        raise RuntimeError(
            f"NanoQuant завершился с кодом {exit_code}. Последние строки настоящей ошибки ниже; "
            f"полный лог: {LOCAL_LOG}\n\n" + "\n".join(tail_lines)
        )

    if not LOCAL_OUTPUT.is_file() or LOCAL_OUTPUT.stat().st_size == 0:
        display(FileLink(str(LOCAL_LOG), result_html_prefix="Полный лог: "))
        raise RuntimeError(f"Команда завершилась без итогового чекпойнта. Лог: {LOCAL_LOG}")

    shutil.copy2(LOCAL_OUTPUT, DRIVE_TEMP_OUTPUT)
    if DRIVE_TEMP_OUTPUT.stat().st_size != LOCAL_OUTPUT.stat().st_size:
        DRIVE_TEMP_OUTPUT.unlink(missing_ok=True)
        raise IOError("Размер файла после копирования в Drive не совпал с исходным")
    DRIVE_TEMP_OUTPUT.replace(DRIVE_OUTPUT)

    run_metadata = {
        "finished_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "profile": PROFILE_RELATIVE_PATH,
        "bits": 0.55,
        "calibration_windows": 128,
        "sequence_length": 2048,
        "attention_backend": attention_backend,
        "gpu": props.name,
        "gpu_vram_gib": round(gpu_vram_gib, 2),
        "pytorch": torch.__version__,
        "cuda_runtime": cuda_version,
        "nanoquant_commit": GIT_COMMIT,
        "checkpoint_file": DRIVE_OUTPUT.name,
        "checkpoint_size_bytes": DRIVE_OUTPUT.stat().st_size,
    }
    temp_metadata = DRIVE_RUN_DIR / "run_complete.json.tmp"
    temp_metadata.write_text(json.dumps(run_metadata, indent=2) + "\n")
    temp_metadata.replace(DRIVE_COMPLETE)

    print("Квантование завершено.")
    print(f"Чекпойнт: {DRIVE_OUTPUT}")
    print(f"Метаданные: {DRIVE_COMPLETE}")
    display(FileLink(str(DRIVE_LOG), result_html_prefix="Лог запуска: "))


### Что будет в Google Drive

В папке `MyDrive/NanoQuant/runs/qwen38-27b-0.55-balanced-uniform-bounded/` появятся:

- `Qwen3.8-27B-NQ-0.55-balanced-uniform-bounded.pt` — итоговый чекпойнт;
- `run_complete.json` — профиль, ревизии кода и модели, GPU и параметры запуска;
- `nanoquant.log` — полный лог квантования;
- `bf16-nanoquant-comparison.log` — полный лог парной оценки;
- `bf16-nanoquant-comparison.previous.log` — копия прошлого лога для восстановления метрик после сбоя;
- `qwen38-0.55-balanced-uniform-bounded-eval.json` — конфигурация оценки;
- `bf16-nanoquant-comparison.json` — исходные метрики обеих моделей и рассчитанные разницы.

Открыть этот ноутбук позднее можно из форка: [NanoQuant Colab notebook](https://colab.research.google.com/github/rkuznetsovcg/NanoQuant/blob/main/notebooks/qwen38_0.55_colab.ipynb).


## Парная оценка BF16 и NanoQuant — свежий повторный прогон

Для готового чекпойнта запускай эту ячейку. Она сама обновит код из нашего форка и подключит Google Drive, если он ещё не подключён. В текущем runtime должны быть установлены зависимости из ячейки установки. Новое квантование здесь не запускается.

По умолчанию `EVAL_FRESH = True`: обе модели оцениваются заново, предыдущие отчёт и лог сохраняются в `previous-evaluations/`. Для продолжения прерванной оценки поставь `False`; результаты переиспользуются только при совпадении кода, версий, данных и настроек. Старые результаты с ошибками подсчёта не переиспользуются.

Перед использованием GPU проверяются все ключи, формы и числа готового `.pt`, включая ранги и масштабные коэффициенты. Отсутствующие, лишние или некорректные веса останавливают оценку с понятной ошибкой. Проверка формата записывается в итоговый JSON.

Затем по очереди загружаются BF16 и NanoQuant. Используются одна закреплённая ревизия Hugging Face, один токенизатор, WikiText-2 raw test, наши 128 калибровочных окон и шесть zero-shot задач: BoolQ, PIQA, HellaSwag, WinoGrande, ARC Easy, ARC Challenge. Контекст явно ограничен 2048 токенами у обеих моделей, few-shot = 0, seed = 0. Batch size подбирается автоматически под память каждой модели; между моделями память GPU очищается. PPL суммируется в FP32 с целочисленным подсчётом токенов и исключением padding.

Отчёт содержит отдельные таблицы **PPL**, **KL** и **zero-shot**. WikiText-2 test — независимый срез. Калибровочные окна использовались для настройки и помечены как внутривыборочная диагностика. KL считается как `KL(p_BF16 || p_NanoQuant)` по полному словарю на каждом 64-м предсказании WikiText-2; контекст — 2048. Единица — nats на токен; меньше означает ближе к BF16. BF16-распределения временно занимают несколько гигабайт локального диска; после успешного завершения кэш удаляется.

KL измеряет расхождение, а качество задач показывают их собственные метрики. См. [Displacement Is Not Direction](https://arxiv.org/abs/2606.19558). Прогресс и полный вывод процесса видны под ячейкой; отчёт и лог сохраняются в ту же папку Google Drive, где лежит чекпойнт.


In [ ]:
import datetime
import html
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path
from IPython.display import FileLink, HTML, display

import torch

# True пересчитывает обе модели. False продолжает прерванную оценку с тем же протоколом.
EVAL_FRESH = True
EVAL_REPO_URL = "https://github.com/rkuznetsovcg/NanoQuant.git"
MODEL_ID = globals().get("MODEL_ID", "Qwen/Qwen3.8-27B")
PROFILE_RELATIVE_PATH = globals().get("PROFILE_RELATIVE_PATH", "configs/qwen38-0.55-balanced.json")

EVAL_REPO = Path("/content/NanoQuant")
EVAL_RUN_DIR = Path("/content/drive/MyDrive/NanoQuant/runs/qwen38-27b-0.55-balanced-uniform-bounded")
EVAL_CHECKPOINT = EVAL_RUN_DIR / "Qwen3.8-27B-NQ-0.55-balanced-uniform-bounded.pt"
EVAL_METADATA = EVAL_RUN_DIR / "run_complete.json"
EVAL_COMPARISON = EVAL_RUN_DIR / "bf16-nanoquant-comparison.json"
EVAL_LOG = EVAL_RUN_DIR / "bf16-nanoquant-comparison.log"
EVAL_PREVIOUS_LOG = EVAL_RUN_DIR / "bf16-nanoquant-comparison.previous.log"

if not torch.cuda.is_available():
    raise RuntimeError("Для сравнения 27B выбери Colab GPU с достаточной памятью и установи зависимости из ячейки выше.")
if not Path("/content/drive/MyDrive").is_dir():
    from google.colab import drive
    drive.mount("/content/drive")

# Update the source even when only this cell is copied into an older notebook.
# Each evaluation runs in a new Python process, so old notebook imports cannot hide fixes.
if not EVAL_REPO.exists():
    subprocess.run(["git", "clone", "--branch", "main", "--single-branch", EVAL_REPO_URL, str(EVAL_REPO)], check=True)
elif (EVAL_REPO / ".git").is_dir():
    subprocess.run(["git", "-C", str(EVAL_REPO), "pull", "--ff-only", EVAL_REPO_URL, "main"], check=True)
else:
    raise RuntimeError(f"{EVAL_REPO} уже существует и не является Git-репозиторием; укажи папку установленного форка.")
print("Код оценки:", subprocess.check_output(["git", "-C", str(EVAL_REPO), "rev-parse", "HEAD"], text=True).strip())
if not EVAL_CHECKPOINT.is_file() or not EVAL_METADATA.is_file():
    raise FileNotFoundError("В Google Drive не найдены чекпойнт и run_complete.json. Сначала заверши квантование.")

metadata = json.loads(EVAL_METADATA.read_text())
expected_size = int(metadata.get("checkpoint_size_bytes", 0))
actual_size = EVAL_CHECKPOINT.stat().st_size
if metadata.get("model_id") != MODEL_ID or (expected_size and actual_size != expected_size):
    raise ValueError(
        f"Чекпойнт не совпадает с метаданными: модель={metadata.get('model_id')}, "
        f"размер={actual_size}, ожидался={expected_size}"
    )

eval_profile = json.loads((EVAL_REPO / PROFILE_RELATIVE_PATH).read_text())
calibration_path = (EVAL_REPO / eval_profile["calib_dataset"]).resolve()
if not calibration_path.is_dir():
    raise FileNotFoundError(f"Не найден калибровочный набор: {calibration_path}")

# Prefer an immutable SHA saved by quantization or preflight. Older metadata may
# lack it; resolve the current Hub SHA directly, with a local-cache fallback.
import re


def is_commit_sha(value):
    return isinstance(value, str) and re.fullmatch(r"[0-9a-f]{40,64}", value) is not None


recorded_revision = metadata.get("model_revision")
preflight_revision = globals().get("MODEL_REVISION")
previous_protocol = {}
if EVAL_COMPARISON.is_file():
    previous_protocol = json.loads(EVAL_COMPARISON.read_text()).get("protocol", {})
previous_revision = previous_protocol.get("model_revision") if previous_protocol.get("model_id") == MODEL_ID else None


def resolve_evaluation_revision(model_id):
    from huggingface_hub import HfApi, try_to_load_from_cache
    hub_error = None
    try:
        revision = HfApi().model_info(model_id, revision="main").sha
        if is_commit_sha(revision):
            return revision, "Hugging Face Hub model_info"
    except Exception as error:
        hub_error = error
    cached = try_to_load_from_cache(model_id, "config.json", revision="main")
    if isinstance(cached, str) and is_commit_sha(Path(cached).resolve().parent.name):
        return Path(cached).resolve().parent.name, "local Hugging Face snapshot cache"
    raise RuntimeError(f"Не удалось закрепить ревизию {model_id}: {hub_error}")

if is_commit_sha(recorded_revision):
    model_revision = recorded_revision
    revision_source = "run_complete.json"
elif is_commit_sha(previous_revision):
    model_revision = previous_revision
    revision_source = "previous paired comparison protocol"
elif is_commit_sha(preflight_revision):
    model_revision = preflight_revision
    revision_source = globals().get("MODEL_REVISION_SOURCE", "Colab preflight config")
else:
    model_revision, revision_source = resolve_evaluation_revision(MODEL_ID)
    revision_source = f"resolved during evaluation: {revision_source}"

eval_profile.update({
    "model_id": MODEL_ID,
    "calib_dataset": str(calibration_path),
    "attn_implementation": metadata.get("attention_backend", "sdpa"),
    "ppl_task": "wikitext2,calibration",
    "zeroshot_task": "boolq,piqa,hellaswag,winogrande,arc_easy,arc_challenge",
    "num_fewshot": 0,
    "limit": -1,
    "batch_size": "auto",
})
EVAL_CONFIG = EVAL_RUN_DIR / "qwen38-0.55-balanced-uniform-bounded-eval.json"
if EVAL_FRESH:
    archive_dir = EVAL_RUN_DIR / "previous-evaluations" / datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    previous_files = [path for path in (EVAL_COMPARISON, EVAL_LOG, EVAL_CONFIG) if path.is_file()]
    if previous_files:
        archive_dir.mkdir(parents=True, exist_ok=True)
        for previous_file in previous_files:
            shutil.copy2(previous_file, archive_dir / previous_file.name)
        print(f"Предыдущие результаты сохранены: {archive_dir}")
EVAL_CONFIG.write_text(json.dumps(eval_profile, indent=2) + "\n")

eval_env = os.environ.copy()
eval_source = str(EVAL_REPO / "src")
old_pythonpath = eval_env.get("PYTHONPATH", "")
eval_env["PYTHONPATH"] = eval_source + (os.pathsep + old_pythonpath if old_pythonpath else "")
eval_env["PYTHONUNBUFFERED"] = "1"
eval_command = [
    sys.executable, "-u", "-m", "nanoquant.compare_eval",
    "--config", str(EVAL_CONFIG),
    "--checkpoint", str(EVAL_CHECKPOINT),
    "--run-metadata", str(EVAL_METADATA),
    "--output", str(EVAL_COMPARISON),
    "--revision", model_revision,
    "--revision-source", revision_source,
    "--resume-log", str(EVAL_PREVIOUS_LOG),
]
if EVAL_FRESH:
    eval_command.append("--fresh")
print("Сначала проверяю формат и все веса готового чекпойнта; затем сравниваю обе модели…")
print("Режим: " + ("свежий прогон BF16 + NanoQuant" if EVAL_FRESH else "продолжение совместимой оценки"))
print(f"Ревизия модели: {model_revision} ({revision_source})")
print(f"Чекпойнт: {EVAL_CHECKPOINT} ({actual_size / (1024 ** 3):.2f} GiB)")
print("GPU будет занят одной моделью за раз; после BF16 веса выгружаются перед загрузкой NanoQuant.")
print(f"Полный лог: {EVAL_LOG}")
if EVAL_LOG.is_file() and not EVAL_FRESH:
    shutil.copy2(EVAL_LOG, EVAL_PREVIOUS_LOG)
    print(f"Предыдущий лог сохранён для продолжения: {EVAL_PREVIOUS_LOG}")

process = subprocess.Popen(
    eval_command,
    cwd=EVAL_REPO,
    env=eval_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=8192,
)
try:
    with EVAL_LOG.open("wb") as log_file:
        while True:
            chunk = process.stdout.read1(8192)
            if not chunk:
                break
            sys.stdout.write(chunk.decode("utf-8", errors="replace"))
            sys.stdout.flush()
            log_file.write(chunk)
            log_file.flush()
    return_code = process.wait()
finally:
    if process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
if return_code != 0:
    tail = EVAL_LOG.read_text(errors="replace").replace("\r", "\n").splitlines()[-35:]
    display(FileLink(str(EVAL_LOG), result_html_prefix="Полный лог: "))
    raise RuntimeError(f"Парная оценка завершилась с кодом {return_code}. Последние сообщения:\n" + "\n".join(tail))

comparison = json.loads(EVAL_COMPARISON.read_text())
if comparison.get("status") != "complete":
    raise RuntimeError(f"Нет завершённого парного отчёта: {EVAL_COMPARISON}")

audit = comparison["checkpoint_audit"]
print(f"Проверка чекпойнта: {audit['status']}; сжатых слоёв: {audit['quantized_linear_count']}; "
      f"фактический бюджет выбранных линейных весов: {audit['selected_linears_bpw']:.3f} bpw")

rows = []
for dataset, values in comparison["comparison"]["perplexity"].items():
    label = "WikiText-2 test" if dataset == "wikitext2" else "Калибровочные окна (in-sample)"
    rows.append(
        "<tr><td>" + html.escape(label) + "</td>"
        f"<td>{values['bf16']:.4f}</td><td>{values['nanoquant']:.4f}</td>"
        f"<td>{values['delta_nanoquant_minus_bf16']:+.4f} ({values['relative_change_percent']:+.2f}%)</td></tr>"
    )
ppl_table = (
    "<h3>PPL — ниже лучше</h3><table><tr><th>Набор</th><th>BF16</th>"
    "<th>NanoQuant</th><th>Δ (NQ − BF16)</th></tr>" + "".join(rows) + "</table>"
)

kl_rows = []
for dataset, values in comparison["comparison"]["kl_divergence"].items():
    kl_rows.append(
        "<tr><td>" + html.escape(values["dataset"]) + "</td>"
        + "<td>" + html.escape(values["direction"]) + "</td>"
        + f"<td>{values['mean_nats_per_token']:.6g}</td>"
        + f"<td>{values['median_nats_per_token']:.6g}</td>"
        + f"<td>{values['p90_nats_per_token']:.6g}</td>"
        + f"<td>{values['predicted_positions']}</td></tr>"
    )
kl_protocol = comparison["protocol"]["kl_divergence"]
kl_table = (
    "<h3>KL-дивергенция — ниже означает ближе к BF16</h3>"
    "<p>Полный словарь; выборка позиций через каждые "
    + str(kl_protocol["position_stride"])
    + " токенов; контекст " + str(kl_protocol["context_length"])
    + " токенов. Единица: nats на токен.</p>"
    "<table><tr><th>Набор</th><th>Направление</th><th>Среднее</th>"
    "<th>Медиана</th><th>p90</th><th>Позиций</th></tr>"
    + "".join(kl_rows) + "</table>"
    "<p>KL измеряет расхождение с BF16, а не качество задачи. По статье arXiv:2606.19558, "
    "среди близких к BF16 вариантов она не даёт надёжного рейтинга downstream-качества.</p>"
)

task_rows = []
for task, values in comparison["comparison"]["zero_shot"].items():
    task_rows.append(
        "<tr><td>" + html.escape(task) + "</td><td>" + html.escape(values["metric"]) + "</td>"
        f"<td>{values['bf16'] * 100:.2f}%</td><td>{values['nanoquant'] * 100:.2f}%</td>"
        f"<td>{values['delta_percentage_points']:+.2f} pp</td></tr>"
    )
task_table = (
    "<h3>Zero-shot — выше лучше</h3><table><tr><th>Задача</th><th>Метрика</th>"
    "<th>BF16</th><th>NanoQuant</th><th>Δ</th></tr>" + "".join(task_rows) + "</table>"
)
mean_delta = comparison["comparison"]["mean_zero_shot_delta_percentage_points"]
revision_note = (
    "Ревизия исходной модели записана вместе с запуском квантования."
    if comparison["protocol"]["revision_recorded_during_quantization"]
    else "В старом run_complete.json ревизия модели не была записана; обе оценки закреплены на "
         "ревизии, разрешённой для этой оценки."
)
display(HTML(
    "<div style='font:14px system-ui;max-width:1000px'>"
    "<h2>Сравнение BF16 и NanoQuant</h2>"
    f"<p>Среднее изменение по шести zero-shot задачам: <b>{mean_delta:+.2f} pp</b>. "
    "Положительное значение означает преимущество NanoQuant.</p>"
    + ppl_table + kl_table + task_table
    + "<p>" + html.escape(revision_note) + "</p>"
    + "</div>"
))
print(f"Парный отчёт: {EVAL_COMPARISON}")
display(FileLink(str(EVAL_COMPARISON), result_html_prefix="JSON-отчёт: "))
display(FileLink(str(EVAL_LOG), result_html_prefix="Полный лог: "))
